# NB3 - ICDAR 2015 Baselines

Trains YOLOv8s and fine-tunes YOLO-World on ICDAR 2015 Incidental Scene Text,
runs a zero-shot prompt sweep, optionally trains RT-DETR-l, and pushes
`ausdm-v3-icdar` for NB5. This is the cross-dataset arm: it shows the container
findings are not specific to one dataset.

**Kaggle setup:** phone-verified account, internet on, dataset
`icdar2015-task41-raw` mounted (the four rrc.cvc.uab.es Challenge 4 archives).
No Roboflow key needed here. The polygon-to-axis-aligned conversion is re-run
from the raw archives, identical to NB0.

**Runtime ~2.9 h GPU** (YOLOv8s ~1.2 h, YOLO-World fine-tune ~1.5 h + sweep
~10 min). With `RUN_RTDETR=True` add ~1.7 h for the droppable RT-DETR-l run.
Use Save Version - Save and Run All (Commit).


## 0. One-time environment setup

In [ ]:
# Same pins as the other notebooks. Uninstall ray to avoid the broken raytune callback.
! pip install -q ultralytics==8.3.40 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true

## 1. Imports, constants, seeds

In [ ]:
# Standard library
import json, os, sys, shutil, subprocess, time, random, zipfile
from pathlib import Path

# Block ultralytics 8.3.40's broken raytune callback
sys.modules['ray'] = None

# Numeric and plotting
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# Deep learning
import torch

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
DATA_ROOT      = KAGGLE_WORKING / 'data'
ICDAR_DIR      = DATA_ROOT / 'icdar2015'
RUNS_DIR       = KAGGLE_WORKING / 'runs'
RESULTS_DIR    = KAGGLE_WORKING / 'results'
STATE_FILE     = KAGGLE_WORKING / 'nb3_state.json'

# Training config (plan section 5.1)
IMGSZ         = 640
EPOCHS        = 100
EPOCHS_RTDETR = 50           # plan 5.1: ICDAR RT-DETR runs 50 epochs
BATCH         = 16           # YOLOv8s / YOLO-World
BATCH_RTDETR  = 8            # avoids OOM on T4
OPTIMIZER     = 'auto'
LR0, LRF      = 0.01, 0.01
PATIENCE      = 50
DEVICE        = 0

YOLO_WEIGHTS   = 'yolov8s.pt'
WORLD_WEIGHTS  = 'yolov8s-worldv2.pt'
RTDETR_WEIGHTS = 'rtdetr-l.pt'

# RT-DETR is the designated sacrifice (plan 5). Leave off unless Day 1 runs ahead.
RUN_RTDETR = False

# Zero-shot prompts for scene text. Prompt 1 is the class-name baseline.
PROMPTS = ['text', 'word', 'scene text', 'printed text']

DATASET_SLUG = 'ausdm-v3-icdar'
KAGGLE_USERNAME_OVERRIDE = ''   # set if auto-detect returns 'unknown'

for d in [DATA_ROOT, ICDAR_DIR, RUNS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'KAGGLE_WORKING = {KAGGLE_WORKING}')
print(f'RUN_RTDETR     = {RUN_RTDETR}')
print(f'prompts        = {PROMPTS}')
print(f'SEED           = {SEED}')

## 2. Resumable state and restore

Restore from a previously pushed `ausdm-v3-icdar` mount so completed stages skip
on a re-run.


In [ ]:
# _load_state / _mark_done / _is_done - stage-level idempotency across sessions
def _load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def _mark_done(stage, extra=None):
    st = _load_state()
    st[stage] = {'done': True, 'ts': time.time()}
    if extra:
        st[stage].update(extra)
    STATE_FILE.write_text(json.dumps(st, indent=2))

def _is_done(stage):
    return _load_state().get(stage, {}).get('done', False)

# _restore_from_prior_push - pull a prior NB3 output back into working
def _restore_from_prior_push():
    if STATE_FILE.exists():
        return 'local state already present'
    if not KAGGLE_INPUT.exists():
        return 'no /kaggle/input'
    for cand in KAGGLE_INPUT.rglob('nb3_state.json'):
        src = cand.parent
        shutil.copy2(cand, STATE_FILE)
        for f in ['icdar_metrics.csv', 'icdar_prompt_sweep.csv']:
            if (src / f).exists():
                shutil.copy2(src / f, RESULTS_DIR / f)
        for sub in ['weights', 'figures']:
            if (src / sub).exists():
                shutil.copytree(src / sub, RESULTS_DIR / sub, dirs_exist_ok=True)
        return f'restored from {src}'
    return 'no prior push mounted'

print('restore:', _restore_from_prior_push())
print('current state:', json.dumps(_load_state(), indent=2))

## 3. Kaggle username and ICDAR mount - fail before any compute

In [ ]:
# _kaggle_user - try every credential source Kaggle might expose, in order
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u:
            return u
    except Exception:
        pass
    u = os.environ.get('KAGGLE_USERNAME')
    if u:
        return u
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            return json.loads(kj.read_text())['username']
        except Exception:
            pass
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i + 1 < len(parts):
                    return parts[i + 1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
print('Kaggle username:', KAGGLE_USER)

# _find_icdar_mount - locate icdar2015-task41-raw wherever Kaggle placed it
def _find_icdar_mount():
    if not KAGGLE_INPUT.exists():
        return None
    def _has_marker(d):
        return ((d / 'ch4_training_images').exists() or
                (d / 'ch4_training_images.zip').exists())
    exact = KAGGLE_INPUT / 'icdar2015-task41-raw'
    if exact.exists() and _has_marker(exact):
        return exact
    for root, dirs, _files in os.walk(KAGGLE_INPUT):
        depth = Path(root).relative_to(KAGGLE_INPUT).parts
        if len(depth) > 3:
            dirs[:] = []; continue
        if _has_marker(Path(root)):
            return Path(root)
    return None

ICDAR_MOUNT = _find_icdar_mount()
assert ICDAR_MOUNT is not None, (
    'ICDAR raw dataset not mounted. Add Data -> "icdar2015-task41-raw" -> Add, '
    f'then re-run. /kaggle/input currently: '
    f'{[p.name for p in KAGGLE_INPUT.iterdir()] if KAGGLE_INPUT.exists() else "nothing"}')
print('ICDAR mount   :', ICDAR_MOUNT)
if KAGGLE_USER == 'unknown':
    print('WARNING: set KAGGLE_USERNAME_OVERRIDE in section 1 before the dry-run push.')

## 4. Rebuild ICDAR 2015 - unpack and polygon-to-axis-aligned conversion

Same conversion as NB0. Training GT is `x1,y1,...,x4,y4,transcription`; test GT
has only the eight coordinate columns. `###` transcriptions are illegible
don't-care regions and are dropped. Boxes are the tight axis-aligned box around
each quad. Expected output: 1000 train images, 500 test images.


In [ ]:
# _unpack_icdar - handle either raw zips or Kaggle-auto-extracted folders
TEST_GT_CAND_ZIP = ['Challenge4_Test_Task1_GT.zip','Challenge4_Test_Task4_GT.zip',
                    'ch4_test_localization_transcription_gt.zip','ch4_test_gt.zip']
TEST_GT_CAND_DIR = ['Challenge4_Test_Task1_GT','Challenge4_Test_Task4_GT',
                    'ch4_test_localization_transcription_gt','ch4_test_gt']

def _unpack_icdar():
    zips = {'train_images':'ch4_training_images.zip',
            'train_gt':'ch4_training_localization_transcription_gt.zip',
            'test_images':'ch4_test_images.zip', 'test_gt':None}
    for cand in TEST_GT_CAND_ZIP:
        if (ICDAR_MOUNT / cand).exists():
            zips['test_gt'] = cand; break
    any_zip = any((ICDAR_MOUNT / v).exists() for v in zips.values() if v)
    if not any_zip:
        dirs = {'train_images':'ch4_training_images',
                'train_gt':'ch4_training_localization_transcription_gt',
                'test_images':'ch4_test_images', 'test_gt':None}
        for cand in TEST_GT_CAND_DIR:
            if (ICDAR_MOUNT / cand).is_dir():
                dirs['test_gt'] = cand; break
        assert dirs['test_gt'], 'No test GT folder or zip found in mount'
        for k, d in dirs.items():
            assert (ICDAR_MOUNT / d).is_dir(), f'Missing folder: {d}'
        return ICDAR_MOUNT, dirs
    assert zips['test_gt'], 'Test GT zip not found - check upload naming'
    stage_dir = KAGGLE_WORKING / 'icdar_raw'; stage_dir.mkdir(exist_ok=True)
    for key, name in zips.items():
        out = stage_dir / key
        if out.exists() and any(out.iterdir()):
            continue
        out.mkdir(exist_ok=True)
        with zipfile.ZipFile(ICDAR_MOUNT / name) as z:
            z.extractall(out)
    return stage_dir, {k: k for k in zips}

# _find_dir_with - locate the actual images/gt root inside an archive
def _find_dir_with(root, glob_pat):
    if any(root.glob(glob_pat)):
        return root
    for p in root.rglob('*'):
        if p.is_dir() and any(p.glob(glob_pat)):
            return p
    return root

# _convert_pair - polygons -> axis-aligned YOLO labels; drop '###' transcriptions
def _convert_pair(img_dir, gt_dir, out_split):
    (ICDAR_DIR / out_split / 'images').mkdir(parents=True, exist_ok=True)
    (ICDAR_DIR / out_split / 'labels').mkdir(parents=True, exist_ok=True)
    n_img = n_box = n_drop = 0
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
        gt = gt_dir / f'gt_{img.stem}.txt'
        if not gt.exists(): continue
        text = gt.read_text(encoding='utf-8-sig', errors='ignore')
        im = Image.open(img); W, H = im.size
        lines_out = []
        for ln in text.splitlines():
            if not ln.strip(): continue
            parts = ln.split(',', 8)
            if len(parts) < 8: continue
            try:
                xs = [float(parts[i]) for i in (0,2,4,6)]
                ys = [float(parts[i]) for i in (1,3,5,7)]
            except ValueError: continue
            trans = parts[8].strip() if len(parts) >= 9 else ''
            if trans == '###':
                n_drop += 1; continue
            x_min, x_max = max(0, min(xs)), min(W, max(xs))
            y_min, y_max = max(0, min(ys)), min(H, max(ys))
            bw = x_max - x_min; bh = y_max - y_min
            if bw <= 1 or bh <= 1: continue
            cx = (x_min + x_max)/2/W; cy = (y_min + y_max)/2/H
            lines_out.append(f'0 {cx:.6f} {cy:.6f} {bw/W:.6f} {bh/H:.6f}')
            n_box += 1
        (ICDAR_DIR / out_split / 'labels' / f'{img.stem}.txt').write_text(
            '\n'.join(lines_out) + ('\n' if lines_out else ''))
        shutil.copy2(img, ICDAR_DIR / out_split / 'images' / img.name)
        n_img += 1
    return n_img, n_box, n_drop

if not _is_done('icdar_convert'):
    ICDAR_RAW, SUB = _unpack_icdar()
    tr_img = _find_dir_with(ICDAR_RAW / SUB['train_images'], '*.jpg')
    tr_gt  = _find_dir_with(ICDAR_RAW / SUB['train_gt'],     'gt_*.txt')
    te_img = _find_dir_with(ICDAR_RAW / SUB['test_images'],  '*.jpg')
    te_gt  = _find_dir_with(ICDAR_RAW / SUB['test_gt'],      'gt_*.txt')
    tr = _convert_pair(tr_img, tr_gt, 'train')
    te = _convert_pair(te_img, te_gt, 'test')
    print(f'train: {tr[0]} imgs, {tr[1]} boxes, {tr[2]} dropped')
    print(f'test : {te[0]} imgs, {te[1]} boxes, {te[2]} dropped')
    assert tr[0] >= 900 and te[0] >= 450, f'Unexpected counts: train {tr[0]}, test {te[0]}'
    _mark_done('icdar_convert', {'train_imgs': tr[0], 'test_imgs': te[0],
                                 'train_boxes': tr[1], 'test_boxes': te[1]})
else:
    print('SKIP icdar_convert:', _load_state()['icdar_convert'])

ICDAR_YAML = ICDAR_DIR / 'data.yaml'
ICDAR_YAML.write_text(f'''path: {ICDAR_DIR}
train: train/images
val:   test/images
test:  test/images
nc: 1
names: ['text']
''')
print('icdar data.yaml:', ICDAR_YAML,
      '| test images:', len(list((ICDAR_DIR/'test'/'images').iterdir())))

## 5. Neutralise the raytune callback

In [ ]:
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}
    _rt.tune = None
    print('raytune callback neutralised (code swapped to no-op)')
except Exception as e:
    print('raytune patch skipped:', e)

## 6. Authenticated dry-run push

Catch owner-id / permission problems before the multi-hour training.


In [ ]:
PUSH_DIR = KAGGLE_WORKING / 'push_stage'

# _stage_push_dir - rebuild staging folder and metadata
def _stage_push_dir(files=None, dirs=None):
    if PUSH_DIR.exists():
        shutil.rmtree(PUSH_DIR)
    PUSH_DIR.mkdir(parents=True, exist_ok=True)
    for f in (files or []):
        if Path(f).exists():
            shutil.copy2(f, PUSH_DIR / Path(f).name)
    for d in (dirs or []):
        if Path(d).exists():
            shutil.copytree(d, PUSH_DIR / Path(d).name, dirs_exist_ok=True)
    meta = {'title': 'AusDM v3 ICDAR 2015',
            'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
            'licenses': [{'name': 'CC-BY-SA-4.0'}]}
    (PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
    return meta

# _push - create on first run, version-bump thereafter
def _push(message):
    # Robust push. The kaggle CLI can exit 0 while printing 'already in use',
    # so success is judged from the output text, not the return code. If the
    # dataset already exists, version-bump instead of create.
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-220:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())
assert KAGGLE_USER != 'unknown', 'Set KAGGLE_USERNAME_OVERRIDE in section 1 and re-run.'

(RESULTS_DIR / 'nb3_marker.txt').write_text('nb3 dry-run ok\n')
_stage_push_dir(files=[RESULTS_DIR / 'nb3_marker.txt'])
if _push('NB3 dry-run'):
    print('dry-run push OK ->', f'{KAGGLE_USER}/{DATASET_SLUG}')
else:
    raise RuntimeError('Dry-run push failed. Fix auth / username before training.')

## 7. Train YOLOv8s (100 epochs, batch 16)

In [ ]:
YOLO_RUN  = RUNS_DIR / 'yolov8s_icdar'
YOLO_BEST = YOLO_RUN / 'weights' / 'best.pt'

if not _is_done('train_yolov8s') and not YOLO_BEST.exists():
    from ultralytics import YOLO
    model = YOLO(YOLO_WEIGHTS)
    model.train(data=str(ICDAR_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                device=DEVICE, project=str(RUNS_DIR), name='yolov8s_icdar',
                exist_ok=True, verbose=False, plots=True)
    assert YOLO_BEST.exists(), 'YOLOv8s best.pt not written'
    _mark_done('train_yolov8s', {'best': str(YOLO_BEST)})
    print('YOLOv8s training done ->', YOLO_BEST)
else:
    print('SKIP train_yolov8s, best at', YOLO_BEST)

## 8. Evaluate YOLOv8s and checkpoint

In [ ]:
# _eval_test - test-split metrics for a YOLO or RTDETR checkpoint
def _eval_test(weights, kind, name):
    if kind == 'yolo':
        from ultralytics import YOLO; m = YOLO(str(weights))
    else:
        from ultralytics import RTDETR; m = RTDETR(str(weights))
    mt = m.val(data=str(ICDAR_YAML), split='test', imgsz=IMGSZ, device=DEVICE,
               project=str(RUNS_DIR), name=name, exist_ok=True, verbose=False, plots=False)
    return {'mAP50-95': round(float(mt.box.map),4), 'mAP50': round(float(mt.box.map50),4),
            'precision': round(float(mt.box.mp),4), 'recall': round(float(mt.box.mr),4)}

yolo_test = _eval_test(YOLO_BEST, 'yolo', 'yolov8s_test')
yolo_test.update({'model':'yolov8s', 'split':'test'})
print('YOLOv8s test metrics:', json.dumps(yolo_test, indent=2))
_mark_done('eval_yolov8s', {'metrics': yolo_test})

# checkpoint immediately
(RESULTS_DIR / 'weights').mkdir(exist_ok=True)
shutil.copy2(YOLO_BEST, RESULTS_DIR / 'weights' / 'yolov8s_best.pt')
rc = YOLO_RUN / 'results.csv'
if rc.exists(): shutil.copy2(rc, RESULTS_DIR / 'yolov8s_results.csv')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb3_state.json')
_stage_push_dir(dirs=[RESULTS_DIR])
print('YOLOv8s checkpoint push:', 'OK' if _push('NB3 YOLOv8s checkpoint') else 'FAILED')

**8.1 Explanation.**

- ICDAR 2015 is genuinely hard: small, dense, multi-oriented scene text reduced
  to axis-aligned boxes. Expect mAP50-95 well below the container numbers, which
  is the point of including a second, harder dataset.
- No pre-registered gate here; T1 was container-specific. Report the number as-is.


## 9. Fine-tune YOLO-World (100 epochs) and evaluate

In [ ]:
WORLD_RUN  = RUNS_DIR / 'yoloworld_icdar'
WORLD_BEST = WORLD_RUN / 'weights' / 'best.pt'

if not _is_done('train_world') and not WORLD_BEST.exists():
    from ultralytics import YOLOWorld
    model = YOLOWorld(WORLD_WEIGHTS)
    model.set_classes(['text'])
    model.train(data=str(ICDAR_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                device=DEVICE, project=str(RUNS_DIR), name='yoloworld_icdar',
                exist_ok=True, verbose=False, plots=True)
    assert WORLD_BEST.exists(), 'YOLO-World best.pt not written'
    _mark_done('train_world', {'best': str(WORLD_BEST)})
    print('YOLO-World fine-tune done ->', WORLD_BEST)
else:
    print('SKIP train_world, best at', WORLD_BEST)

# _eval_world_test - fine-tuned YOLO-World on the test split
def _eval_world_test(weights, classes, name):
    from ultralytics import YOLOWorld
    m = YOLOWorld(str(weights)); m.set_classes(classes)
    mt = m.val(data=str(ICDAR_YAML), split='test', imgsz=IMGSZ, device=DEVICE,
               project=str(RUNS_DIR), name=name, exist_ok=True, verbose=False, plots=False)
    return {'mAP50-95': round(float(mt.box.map),4), 'mAP50': round(float(mt.box.map50),4),
            'precision': round(float(mt.box.mp),4), 'recall': round(float(mt.box.mr),4)}

world_ft = _eval_world_test(WORLD_BEST, ['text'], 'world_ft_test')
world_ft.update({'model':'yolo-world-ft', 'split':'test'})
print('YOLO-World fine-tuned test metrics:', json.dumps(world_ft, indent=2))
_mark_done('eval_world_ft', {'metrics': world_ft})

shutil.copy2(WORLD_BEST, RESULTS_DIR / 'weights' / 'yoloworld_ft_best.pt')
rc = WORLD_RUN / 'results.csv'
if rc.exists(): shutil.copy2(rc, RESULTS_DIR / 'yoloworld_results.csv')

## 10. Zero-shot prompt sweep on the test split

Pretrained `yolov8s-worldv2.pt`, no fine-tuning. Prompt 1 (`text`) is the
class-name baseline; the rest test scene-text prompt sensitivity, mirroring the
container sweep in NB2.


In [ ]:
# _zero_shot_eval - pretrained YOLO-World, one prompt as the whole vocabulary
def _zero_shot_eval(model, prompt, idx):
    model.set_classes([prompt])
    mt = model.val(data=str(ICDAR_YAML), split='test', imgsz=IMGSZ, device=DEVICE,
                   project=str(RUNS_DIR), name=f'zs_{idx}', exist_ok=True,
                   verbose=False, plots=False)
    return {'idx': idx, 'prompt': prompt,
            'mAP50-95': round(float(mt.box.map),4), 'mAP50': round(float(mt.box.map50),4),
            'precision': round(float(mt.box.mp),4), 'recall': round(float(mt.box.mr),4)}

if not _is_done('prompt_sweep'):
    from ultralytics import YOLOWorld
    zs_model = YOLOWorld(WORLD_WEIGHTS)
    sweep = []
    for i, p in enumerate(PROMPTS, start=1):
        row = _zero_shot_eval(zs_model, p, i)
        print(f'  prompt {i}: {p!r:20s} mAP50-95={row["mAP50-95"]:.4f}  mAP50={row["mAP50"]:.4f}')
        sweep.append(row)
    df_sweep = pd.DataFrame(sweep)
    df_sweep.to_csv(RESULTS_DIR / 'icdar_prompt_sweep.csv', index=False)
    _mark_done('prompt_sweep', {'rows': sweep})
else:
    df_sweep = pd.DataFrame(_load_state()['prompt_sweep']['rows'])
    print('SKIP prompt_sweep')

print()
print(df_sweep[['idx','prompt','mAP50-95','mAP50','precision','recall']].to_string(index=False))

**10.1 Explanation.**

- Scene text is a more natural fit for open-vocabulary detection than a container
  serial, so zero-shot may score higher here than in NB2. Whatever the number,
  report the fine-tuned-versus-zero-shot gap and the prompt spread.


## 11. RT-DETR-l (optional, `RUN_RTDETR`, 50 epochs)

The designated sacrifice (plan section 5). Off by default; set `RUN_RTDETR=True`
in section 1 only if the schedule is ahead.


In [ ]:
rtdetr_test = None
RTDETR_RUN  = RUNS_DIR / 'rtdetr_icdar'
RTDETR_BEST = RTDETR_RUN / 'weights' / 'best.pt'

if RUN_RTDETR:
    if not _is_done('train_rtdetr') and not RTDETR_BEST.exists():
        from ultralytics import RTDETR
        model = RTDETR(RTDETR_WEIGHTS)
        model.train(data=str(ICDAR_YAML), epochs=EPOCHS_RTDETR, imgsz=IMGSZ, batch=BATCH_RTDETR,
                    optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                    device=DEVICE, project=str(RUNS_DIR), name='rtdetr_icdar',
                    exist_ok=True, verbose=False, plots=True)
        assert RTDETR_BEST.exists(), 'RT-DETR-l best.pt not written'
        _mark_done('train_rtdetr', {'best': str(RTDETR_BEST)})
    rtdetr_test = _eval_test(RTDETR_BEST, 'rtdetr', 'rtdetr_test')
    rtdetr_test.update({'model':'rtdetr-l', 'split':'test'})
    print('RT-DETR-l test metrics:', json.dumps(rtdetr_test, indent=2))
    _mark_done('eval_rtdetr', {'metrics': rtdetr_test})
    shutil.copy2(RTDETR_BEST, RESULTS_DIR / 'weights' / 'rtdetr-l_best.pt')
    rc = RTDETR_RUN / 'results.csv'
    if rc.exists(): shutil.copy2(rc, RESULTS_DIR / 'rtdetr-l_results.csv')
else:
    print('RT-DETR skipped (RUN_RTDETR=False). Report the gap honestly in the paper.')

## 12. Consolidated test-split metrics

In [ ]:
# Supervised + fine-tuned rows; add RT-DETR if it ran
rows = [_load_state()['eval_yolov8s']['metrics'],
        _load_state()['eval_world_ft']['metrics']]
if _load_state().get('eval_rtdetr'):
    rows.append(_load_state()['eval_rtdetr']['metrics'])
df_metrics = pd.DataFrame(rows)[['model','split','mAP50-95','mAP50','precision','recall']]
df_metrics.to_csv(RESULTS_DIR / 'icdar_metrics.csv', index=False)
print(df_metrics.to_string(index=False))
print()
best = df_sweep.loc[df_sweep['mAP50-95'].idxmax()]
print(f'best zero-shot prompt: #{int(best["idx"])} {best["prompt"]!r} '
      f'mAP50-95={float(best["mAP50-95"]):.4f}')

**12.1 Explanation.**

- Read the fine-tuned YOLO-World row against the best zero-shot prompt: the gap is
  the cross-dataset counterpart to the container finding in NB2.
- Absolute ICDAR scores are lower than the container numbers; that difference is
  itself reportable as dataset difficulty.


## 13. Prediction sanity figures - boxes on test images

Mandatory box check on the fine-tuned model and the best zero-shot prompt.


In [ ]:
# _draw - draw predicted boxes for a checkpoint (YOLO or YOLO-World) on n test images
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

def _draw(weights, kind, name, classes=None, n=6, conf=0.25):
    if kind == 'world':
        from ultralytics import YOLOWorld; m = YOLOWorld(str(weights)); m.set_classes(classes)
    else:
        from ultralytics import YOLO; m = YOLO(str(weights))
    imgs = sorted((ICDAR_DIR / 'test' / 'images').iterdir())[:n]
    ndet = 0
    for img_path in imgs:
        res = m.predict(str(img_path), imgsz=IMGSZ, conf=conf, device=DEVICE, verbose=False)[0]
        im = Image.open(img_path).convert('RGB'); dr = ImageDraw.Draw(im)
        for b in res.boxes:
            x1,y1,x2,y2 = [float(v) for v in b.xyxy[0]]
            dr.rectangle([x1,y1,x2,y2], outline='lime', width=3); ndet += 1
        im.save(FIG_DIR / f'{name}_{img_path.stem}.png')
    return len(imgs), ndet

best_prompt = df_sweep.loc[df_sweep['mAP50-95'].idxmax()]['prompt']
n1, d1 = _draw(YOLO_BEST,     'yolo',  'pred_yolov8s')
n2, d2 = _draw(WORLD_BEST,    'world', 'pred_world_ft', classes=['text'])
n3, d3 = _draw(WORLD_WEIGHTS, 'world', 'pred_world_zs_best', classes=[best_prompt])
print(f'yolov8s        : {n1} imgs, {d1} boxes')
print(f'world fine-tune: {n2} imgs, {d2} boxes')
print(f'world zero-shot ({best_prompt!r}): {n3} imgs, {d3} boxes')

In [ ]:
# Inline preview so the box check happens in the notebook
from IPython.display import Image as IPImg, display
for p in (sorted(FIG_DIR.glob('pred_yolov8s_*'))[:2] +
          sorted(FIG_DIR.glob('pred_world_zs_best_*'))[:2]):
    print(p.name); display(IPImg(str(p), width=460))

**13.1 Explanation.** Boxes should cluster on scene-text regions. Missed small
or rotated words are expected and worth showing, since they explain the lower
ICDAR scores.


## 14. Final push - `ausdm-v3-icdar`

In [ ]:
_mark_done('final_push')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb3_state.json')
_stage_push_dir(dirs=[RESULTS_DIR])
print('final push:', 'OK' if _push('NB3 final: ICDAR baselines') else 'FAILED')

## 15. Conclusion and handoff to NB5

In [ ]:
print('NB3 state:', json.dumps(_load_state(), indent=2))
print()
print('=== ICDAR test-split metrics ===')
print(pd.read_csv(RESULTS_DIR / 'icdar_metrics.csv').to_string(index=False))
print()
print('=== ICDAR zero-shot prompt sweep ===')
print(pd.read_csv(RESULTS_DIR / 'icdar_prompt_sweep.csv')[
      ['idx','prompt','mAP50-95','mAP50']].to_string(index=False))
print()
print('RT-DETR ran :', bool(_load_state().get('eval_rtdetr')))
print('Pushed      :', f'{KAGGLE_USER}/{DATASET_SLUG}')
print('Next        : NB4 (COCO reference) and NB5 (aggregation)')

**Result.** YOLOv8s and fine-tuned YOLO-World are trained on ICDAR 2015 and
scored on the official 500-image test split, alongside a zero-shot prompt sweep.
RT-DETR-l is optional behind `RUN_RTDETR`. Metrics, weights, the sweep CSV, and
box-drawn figures are pushed as `ausdm-v3-icdar` for NB5.

**Role in the paper.** ICDAR is the harder, cross-dataset check. Lower absolute
scores than the container set are expected and reportable. If RT-DETR is skipped,
state the gap plainly rather than implying it was run.
